### Imports & paths

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
BREAD_PATH = DATA_PATH / "Bread"
PROCESSED_PATH = DATA_PATH / "processed"
MODELS_PATH = PROJECT_ROOT / "models"
RESULTS_PATH = PROJECT_ROOT / "results"

RESULTS_PATH.mkdir(exist_ok=True)

import pandas as pd

PREDICTION_LENGTH = 7  # horizon H
TEST_WINDOW_DAYS = 90  # must be >= H, gives ~13 evaluation weeks
RANDOM_SEED = 123

### Imports from the codebase

In [ ]:
from src.features.encoding import (fit_gln_target_encoding,
                                   apply_gln_te,
                                   fit_gb_id_target_encoding,
                                   apply_gb_id_target_encoding)
from src.models.evaluate import load_model_and_predict, plot_error_analysis

from src.models.hierarchical import evaluate_hierarchical

from src.utils.metrics import calculate_error_metrics
from src.utils.splitting import split_last_n_observations

### Load preprocessed data

In [ ]:
sales_daily = pd.read_parquet(PROCESSED_PATH / "sales_daily.parquet")
store_daily = pd.read_parquet(PROCESSED_PATH / "store_daily.parquet")
share_daily = pd.read_parquet(PROCESSED_PATH / "share_daily.parquet")

### Define test window and split datasets

In [ ]:
store_train, store_test = split_last_n_observations(store_daily, PREDICTION_LENGTH)
share_train, share_test = split_last_n_observations(share_daily, PREDICTION_LENGTH)

### Load trained models

In [ ]:
total_model_path = MODELS_PATH / "xgb_total_7days.pkl"
share_model_path = MODELS_PATH / "xgb_share_7days.pkl"

### Evaluate Total model

In [ ]:
# === Apply SAME encodings as training ===
gln_mapping, gln_global_mean = fit_gln_target_encoding(store_train)

#store_train = apply_gln_te(store_train, gln_mapping, gln_global_mean)
store_test = apply_gln_te(store_test, gln_mapping, gln_global_mean)

In [ ]:
# === Predict using model bundle features ===
y_pred_total = load_model_and_predict(store_test, total_model_path)

y_test_total = store_test["quantity"]

plot_error_analysis(y_test_total, y_pred_total)
calculate_error_metrics(y_test_total, y_pred_total)

### Evaluate Share model

In [ ]:
# gb_id target encoding
mapping_gb, global_mean_gb = fit_gb_id_target_encoding(share_train)

share_test = apply_gb_id_target_encoding(share_test, mapping_gb, global_mean_gb)

# target encoding for gln
mapping, global_mean = fit_gln_target_encoding(share_train)

#share_train = apply_gln_te(share_train, mapping, global_mean)
share_test = apply_gln_te(share_test, mapping, global_mean)

share_test["promo_x_popularity"] = share_test["on_promotion"] * share_test["gb_id_mean_target"]

In [ ]:
store_test["pred_total"] = load_model_and_predict(store_test, total_model_path)

share_test["pred_share"] = load_model_and_predict(share_test, share_model_path)

plot_error_analysis(share_test["target"], share_test["pred_share"])
calculate_error_metrics(share_test["target"], share_test["pred_share"])

### Hierarchical evaluation (core cell)

In [ ]:
hierarchical_metrics, hierarchical_df = evaluate_hierarchical(
    store_test=store_test,
    share_test=share_test,
    total_model_path=total_model_path,
    share_model_path=share_model_path
)